# scRNA Demultiplexing
Vireo-based cell-line demultiplexing results, donor assignment quality, concordance profiles,
and reference-guided validation using Pool_ctr.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import os, glob
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'results/variants.db'
DEMUX_DIR = 'results/demux'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

def load_pool(pool_id):
    path = os.path.join(DEMUX_DIR, pool_id, 'final_assignments.tsv')
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_csv(path, sep='\t')

# Discover all pools that have a final_assignments.tsv
pool_dirs = sorted(glob.glob(os.path.join(DEMUX_DIR, '*/final_assignments.tsv')))
all_pool_ids = [os.path.basename(os.path.dirname(p)) for p in pool_dirs]

print(f'Pools found: {all_pool_ids}')
if not all_pool_ids:
    print('No demux runs found — run the scrna pipeline first.')

## scRNA demultiplexing results
Cell composition, doublet rate, Vireo donor probabilities, and concordance with DB runs.

In [ ]:
if not all_pool_ids:
    print('No demux runs available.')
else:
    pool_select = widgets.Dropdown(
        options=all_pool_ids, value=all_pool_ids[-1],
        description='Pool:', layout=widgets.Layout(width='55%')
    )
    out_pool = widgets.Output()

    def show_pool_demux(pool_id):
        cells = load_pool(pool_id)
        if cells.empty:
            print(f'No data for {pool_id}.')
            return

        n_total   = len(cells)
        n_doublet = (cells.get('source', cells.get('vireo_donor', '')).eq('doublet')).sum()
        # Use 'source' column if present, else fall back to vireo_donor for doublet/unassigned detection
        if 'source' in cells.columns:
            assigned = cells[~cells['source'].isin(['doublet', 'unassigned', 'vireo_only'])]
            doublet_mask = cells['source'] == 'doublet'
        else:
            assigned = cells[~cells['vireo_donor'].isin(['doublet', 'unassigned'])]
            doublet_mask = cells['vireo_donor'] == 'doublet'

        donor_col = 'cell_line' if 'cell_line' in cells.columns else 'vireo_donor'
        donor_counts = cells[donor_col].fillna('unassigned').value_counts()

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                f'Cell composition ({donor_col})',
                'Doublet probability distribution',
                'Max donor probability (assigned cells)',
                'Concordance by donor' if cells.get('concordance', pd.Series()).notna().any() else 'Scorer status'
            ),
            horizontal_spacing=0.12, vertical_spacing=0.15
        )

        # Panel 1: donor composition
        colors = px.colors.qualitative.Safe
        fig.add_trace(go.Bar(
            x=donor_counts.index, y=donor_counts.values,
            marker_color=colors[:len(donor_counts)]
        ), row=1, col=1)

        # Panel 2: doublet probability
        if 'vireo_prob_doublet' in cells.columns:
            fig.add_trace(go.Histogram(
                x=cells['vireo_prob_doublet'].dropna(), nbinsx=50,
                marker_color='#e63946'
            ), row=1, col=2)

        # Panel 3: max probability for assigned cells
        if 'vireo_prob_max' in cells.columns and not assigned.empty:
            for donor in assigned[donor_col].dropna().unique()[:8]:
                sub = assigned[assigned[donor_col] == donor]
                fig.add_trace(go.Histogram(
                    x=sub['vireo_prob_max'], nbinsx=25, name=donor, opacity=0.6
                ), row=2, col=1)
            fig.update_layout(barmode='overlay')

        # Panel 4: concordance or scorer_status
        if 'concordance' in cells.columns and cells['concordance'].notna().any():
            conc = (cells[cells['concordance'].notna()]
                    .groupby(donor_col)['concordance'].mean()
                    .sort_values(ascending=False))
            fig.add_trace(go.Bar(x=conc.index, y=conc.values, marker_color='#2d6a4f'), row=2, col=2)
        elif 'scorer_status' in cells.columns:
            sc_counts = cells['scorer_status'].value_counts()
            fig.add_trace(go.Bar(x=sc_counts.index, y=sc_counts.values, marker_color='#457b9d'), row=2, col=2)

        doublet_n = doublet_mask.sum()
        fig.update_layout(
            height=620,
            title=f'{pool_id}  —  {n_total:,} cells  ({doublet_n} doublets, {doublet_n/n_total*100:.1f}%)',
            showlegend=False
        )
        fig.update_xaxes(tickangle=45, row=1, col=1)
        fig.update_xaxes(tickangle=30, row=2, col=2)
        fig.show()

        summary = (cells.groupby(donor_col, dropna=False)
                   .agg(n_cells=(donor_col, 'count'),
                        mean_prob_max=('vireo_prob_max', 'mean'),
                        mean_concordance=('concordance', 'mean'))
                   .sort_values('n_cells', ascending=False))
        display(summary.style.format({'mean_prob_max': '{:.4f}', 'mean_concordance': '{:.4f}'},
                                     na_rep='—'))

    def on_pool_change(change):
        with out_pool:
            clear_output(wait=True)
            show_pool_demux(change['new'])

    pool_select.observe(on_pool_change, names='value')
    display(pool_select, out_pool)
    show_pool_demux(pool_select.value)

## Concordance profiles — all pools
Distribution of per-cell concordance scores. A clean separation between known same-donor
and different-donor pairs validates the genotype-based assignment.

In [ ]:
if not all_pool_ids:
    print('No pools available.')
else:
    fig = go.Figure()
    for pool_id in all_pool_ids:
        cells = load_pool(pool_id)
        if cells.empty or 'concordance' not in cells.columns:
            continue
        conc = cells['concordance'].dropna()
        if conc.empty:
            continue
        fig.add_trace(go.Violin(
            y=conc, name=pool_id,
            box_visible=True, meanline_visible=True, points='outliers'
        ))
    if fig.data:
        fig.update_layout(
            height=420, title='Concordance distribution per pool',
            yaxis_title='Concordance score', xaxis_tickangle=30
        )
        fig.show()
    else:
        print('No concordance data available in any pool (pools v3+ use guided Vireo without DB scorer).')

## Concordance matrix — Plot A: raw per-cell concordance
Heatmap of mean concordance for each Vireo donor × DB run combination across all pools.

In [ ]:
if not all_pool_ids:
    print('No pools available.')
else:
    all_dfs = []
    for pool_id in all_pool_ids:
        cells = load_pool(pool_id)
        if not cells.empty and 'concordance' in cells.columns and cells['concordance'].notna().any():
            cells['pool_id'] = pool_id
            all_dfs.append(cells)

    if not all_dfs:
        print('No concordance data found across any pool.')
    else:
        all_df = pd.concat(all_dfs, ignore_index=True)
        donor_col = 'cell_line' if 'cell_line' in all_df.columns else 'vireo_donor'

        pivot = (
            all_df[all_df['concordance'].notna()]
            .groupby([donor_col, 'pool_id'])['concordance']
            .mean()
            .unstack(fill_value=np.nan)
        )
        fig = go.Figure(go.Heatmap(
            z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
            colorscale='RdYlGn', zmin=0, zmax=1,
            text=np.round(pivot.values, 3), texttemplate='%{text}'
        ))
        fig.update_layout(
            height=max(350, 30 * len(pivot)),
            title='Plot A: Mean concordance — donor (rows) × pool run (cols)',
            xaxis_tickangle=45
        )
        fig.show()

## Validation — reference-guided Vireo (Pool_ctr)
For the control pool where expected donor identities are known, compares Vireo-assigned
donors against reference to verify assignment quality.

In [ ]:
ctr_pools = [p for p in all_pool_ids if 'ctr' in p.lower() or 'ctrl' in p.lower()]
print(f'Control pools: {ctr_pools}')

for pool_id in ctr_pools:
    cells = load_pool(pool_id)
    if cells.empty:
        print(f'{pool_id}: no data.')
        continue

    donor_col = 'cell_line' if 'cell_line' in cells.columns else 'vireo_donor'
    assigned = cells[~cells.get('source', cells['vireo_donor']).isin(['doublet', 'unassigned', 'vireo_only'])]
    if assigned.empty:
        print(f'{pool_id}: no assigned cells.')
        continue

    print(f'\n--- {pool_id} ({len(cells):,} cells, {len(assigned):,} assigned) ---')

    # Composition summary
    summary = (assigned.groupby(donor_col, dropna=False)
               .agg(n_cells=(donor_col, 'count'),
                    mean_prob=('vireo_prob_max', 'mean'),
                    mean_concordance=('concordance', 'mean'))
               .sort_values('n_cells', ascending=False))
    display(summary.style.format({'mean_prob': '{:.4f}', 'mean_concordance': '{:.4f}'}, na_rep='—'))

    # Agreement bar (if available)
    if 'agreement' in cells.columns:
        agree_counts = cells['agreement'].value_counts()
        agree_pct = agree_counts.get(True, 0) / len(cells) * 100
        print(f'Vireo–scorer agreement: {agree_pct:.1f}% of cells')

    # Concordance bar
    if 'concordance' in cells.columns and cells['concordance'].notna().any():
        conc = (cells[cells['concordance'].notna()]
                .groupby(donor_col)['concordance'].mean()
                .sort_values(ascending=False))
        fig = go.Figure(go.Bar(x=conc.index, y=conc.values, marker_color='#457b9d'))
        fig.update_layout(
            height=320,
            title=f'{pool_id}: mean concordance per assigned donor',
            xaxis_tickangle=45, yaxis_title='Mean concordance'
        )
        fig.show()

## Final assignments summary
Reads `results/demux/final_assignments.tsv` and summarises the per-cell donor calls.

In [ ]:
fa_paths = sorted(glob.glob(os.path.join(DEMUX_DIR, '*/final_assignments.tsv')))
if not fa_paths:
    print('No final_assignments.tsv found.')
else:
    all_fa = []
    for p in fa_paths:
        pool_id = os.path.basename(os.path.dirname(p))
        df = pd.read_csv(p, sep='\t')
        df['pool_id'] = pool_id
        all_fa.append(df)
    fa = pd.concat(all_fa, ignore_index=True)
    donor_col = 'cell_line' if 'cell_line' in fa.columns else 'vireo_donor'

    print(f'Total cells across all pools: {len(fa):,}')
    display(fa.groupby(['pool_id', donor_col], dropna=False)
            .size().rename('n_cells').reset_index()
            .pivot(index=donor_col, columns='pool_id', values='n_cells')
            .fillna(0).astype(int)
            .style.background_gradient(cmap='Blues', axis=None))

    # Summary bar across latest pool only
    latest = all_pool_ids[-1]
    latest_df = load_pool(latest)
    if not latest_df.empty:
        col = 'cell_line' if 'cell_line' in latest_df.columns else 'vireo_donor'
        counts = latest_df[col].fillna('unassigned').value_counts()
        fig = px.bar(x=counts.index, y=counts.values,
                     title=f'Final donor assignments — {latest}',
                     labels={'x': 'Donor', 'y': 'Cell count'})
        fig.update_xaxes(tickangle=45)
        fig.show()